In [22]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [23]:
!pip install -q sentence-transformers faiss-cpu

In [24]:
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer

In [25]:
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [26]:
def combine(df):
    return (
        "Question: " + df["prompt"].fillna("") +
        " A: " + df["A"].fillna("") +
        " B: " + df["B"].fillna("") +
        " C: " + df["C"].fillna("") +
        " D: " + df["D"].fillna("") +
        " E: " + df["E"].fillna("")
    )

train_text = combine(train)
test_text = combine(test)

train_text.iloc[0]

"Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement. C: Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present. D: Martin Heidegger believes that the relations

In [27]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
train_embeddings = embedding_model.encode(
    train_text.tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

test_embeddings = embedding_model.encode(
    test_text.tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

print(train_embeddings.shape)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

(2000, 384)


In [29]:
dimension = train_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(train_embeddings)

print("Vectors stored:", index.ntotal)

Vectors stored: 2000


In [30]:
k = 3

distances, indices = index.search(
    test_embeddings[0].reshape(1, -1),
    k
)

indices

array([[ 721, 1708, 1339]])

In [31]:
print("Current Test Question\n")
print(test.loc[0, "prompt"])

print("\nRetrieved Similar Questions\n")

for idx in indices[0]:
    print(train.loc[idx, "prompt"])
    print("Correct Answer:", train.loc[idx, "answer"])
    print("-" * 60)

Current Test Question

Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully.

Retrieved Similar Questions

Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully.
Correct Answer: A
------------------------------------------------------------
Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully.
Correct Answer: A
------------------------------------------------------------
Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics?
Correct Answer: A
------------------------------------------------------------


In [32]:
retrieved_context = ""

for idx in indices[0]:
    retrieved_context += (
        "Related Question: "
        + train.loc[idx, "prompt"]
        + "\nCorrect Answer: "
        + train.loc[idx, "answer"]
        + "\n\n"
    )

augmented_prompt = f"""
Retrieved Context:

{retrieved_context}

Current Question:

{test.loc[0,'prompt']}

Options:

A. {test.loc[0,'A']}
B. {test.loc[0,'B']}
C. {test.loc[0,'C']}
D. {test.loc[0,'D']}
E. {test.loc[0,'E']}
"""

print(augmented_prompt)


Retrieved Context:

Related Question: Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully.
Correct Answer: A

Related Question: Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully.
Correct Answer: A

Related Question: Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics?
Correct Answer: A



Current Question:

Pick the best possible answer: What is the relationship between the Hamiltonians and eigenstates in supersymmetric quantum mechanics? carefully.

Options:

A. For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with the same energy.
B. For every eigenstate of one Hamiltonian, its partner Hamiltonian has a corresponding eigenstate with a higher energy.
C. For every eigenstate o

In [33]:
k = 3

all_context = []

for embedding in test_embeddings:
    distances, indices = index.search(
        embedding.reshape(1, -1),
        k
    )

    context = ""

    for idx in indices[0]:
        context += (
            train.loc[idx, "prompt"]
            + " "
            + train.loc[idx, "answer"]
            + " "
        )

    all_context.append(context)

In [34]:
augmented_test = test.copy()

augmented_test["retrieved_context"] = all_context

augmented_test.head()

,id,prompt,A,B,C,D,E,retrieved_context
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...",Pick the best possible answer: What is the rel...
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi...","What is the estimated redshift of CEERS-93316,..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...,Pick the best possible answer: What is the rea...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,What is the significance of the redshift-dista...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,What is the Landau-Lifshitz-Gilbert equation u...
